# CodeOrbit on SWE-bench, with a GPU

Runs `scripts/swebench.py` with embeddings on Colab's free T4 GPU, so the
**dense**, **hybrid** and full **codeorbit** arms can run on real issues.

**Before you start:** `Runtime -> Change runtime type -> T4 GPU`.

Results are written to your Google Drive (`MyDrive/codeorbit-runs`) when it
mounts, otherwise to the session (download them with step 9). On Drive a
disconnect loses nothing: re-run steps 1-4, then step 7 with `--resume`.
Free-tier sessions end after a few idle minutes without the tab open and after
at most ~12 hours, and a GPU is not always available.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Where results are saved

Tries Google Drive first, so results survive a disconnect. If Drive will not
mount ("credential propagation was unsuccessful"): allow the pop-up, pick the
same Google account as Colab, tick every permission - or use
`File -> Save a copy in Drive` and run the copy. If it still fails, results are
kept in this session instead: download them with step 9 before you leave.

In [ ]:
import os
CHECKOUTS = '/content/checkouts'                     # repositories: fast local disk
try:
    from google.colab import drive
    drive.mount('/content/drive')
    RUNS = '/content/drive/MyDrive/codeorbit-runs'  # results: kept on Drive
    print('Results will be saved to Google Drive:', RUNS)
except Exception as e:
    RUNS = '/content/codeorbit-runs'                # results: this session only
    print(f'Drive not mounted ({e}).')
    print('Results will be saved in this session only:', RUNS)
    print('Download them with step 9 before the session ends.')
os.makedirs(RUNS, exist_ok=True)

## 3. Get CodeOrbit

Clones the public repository, so it runs whatever is pushed to `main`.
The commit printed here is what every result will record.

In [ ]:
%cd /content
!rm -rf /content/codeorbit
!git clone -q --depth 1 https://github.com/Anilll-coder/codeorbit.git /content/codeorbit
%cd /content/codeorbit
!git log -1 --format='%h %s'
!pip install -q -e . tokenizers

## 4. Start Ollama on the GPU

`pciutils`/`lshw` let the installer detect the GPU; `zstd` unpacks it.
The last line should report the model running on **GPU**.

In [ ]:
!apt-get -qq install -y pciutils lshw zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null

import subprocess, time, requests
subprocess.Popen(['ollama', 'serve'], stdout=open('/content/ollama.log', 'w'),
                 stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        requests.get('http://127.0.0.1:11434/api/version', timeout=2); break
    except Exception:
        time.sleep(1)
!ollama pull nomic-embed-text

t = time.perf_counter()
r = requests.post('http://127.0.0.1:11434/api/embed', timeout=300,
                  json={'model': 'nomic-embed-text', 'input': [f'def f{i}(x): return x' for i in range(256)]})
r.raise_for_status()
print(f'256 embeddings in {time.perf_counter() - t:.1f}s')
!ollama ps

## 5. Download SWE-bench Lite

In [ ]:
!python scripts/swebench.py download --split lite
!python scripts/swebench.py list --split lite

## 6. Pilot: 6 small tasks

Should finish in a few minutes. Check that dense / hybrid appear in the table -
if they are missing, embeddings are not live (look at step 4).

In [ ]:
!python scripts/swebench.py run --split lite --repos psf/requests --embed \
    --tokenizer hf:microsoft/Phi-4-mini-instruct --checkouts {CHECKOUTS} --out {RUNS}

## 7. Main run

Pick ONE. Start with the sample; the full run is the reportable result.

If the session dies, re-run steps 1-4, then add `--resume {RUNS}/<run folder>`
to the same command - finished tasks are skipped.

In [ ]:
# 60 random tasks across all repos (same seed = same tasks every time)
!python scripts/swebench.py run --split lite --sample 60 --seed 0 --embed \
    --tokenizer hf:microsoft/Phi-4-mini-instruct --checkouts {CHECKOUTS} --out {RUNS}

In [ ]:
# All 300 tasks - several hours; keep the tab open
# !python scripts/swebench.py run --split lite --embed \
#     --tokenizer hf:microsoft/Phi-4-mini-instruct --checkouts {CHECKOUTS} --out {RUNS}

In [ ]:
# Resume an interrupted run - fill in the folder name printed when it started
# !python scripts/swebench.py run --split lite --embed \
#     --tokenizer hf:microsoft/Phi-4-mini-instruct --checkouts {CHECKOUTS} --out {RUNS} \
#     --resume {RUNS}/<run folder>

## 8. Results

Function-level table across runs. Each run folder also holds `summary.json`
(including file-level scores) and `per_question.csv`.

To keep a run: download its folder from Drive into `bench/runs/` in your
local repository and commit it there.

In [ ]:
import glob
runs = sorted(glob.glob(f'{RUNS}/*-swebench-*'))
print('\n'.join(runs))
!python scripts/bench_report.py {' '.join(runs)}

## 9. Download the results

Zips every run folder and downloads it to your computer. Unzip into
`bench/runs/` in your local repository to commit a run. Essential if Drive
was not mounted - the session's files are deleted when it ends.

In [ ]:
import shutil
from google.colab import files
zip_path = shutil.make_archive('/content/codeorbit-runs', 'zip', RUNS)
files.download(zip_path)